# N12B — Overtake Battle Sequence Model (Causal TCN)

This notebook trains a temporal sequence model on the same overtake pairs dataset
used in N12, replacing the per-lap snapshot with a **sequence of the last 8 laps**
per pair (X, Y). The architecture is a **Causal TCN** (Temporal Convolutional Network
with left-only padding), which captures gap momentum and battle dynamics without
any look-ahead leakage.

**Why TCN over LSTM:**
- Parallelizable across timesteps (faster training)
- No vanishing gradient for short sequences (5-8 laps)
- Causal convolution is semantically equivalent to masked attention: only past is visible

**Relation to N11 / N12:**
- EDA: fully covered in N11 — not repeated here
- Baseline LightGBM (snapshot): N12 — AUC-PR 0.5491, AUC-ROC 0.8758
- This model: replaces N12 in Strategy Agent inference if AUC-ROC > 0.875

**Input:** `data/processed/overtake_labeled/overtake_pairs_2023_2025.parquet`

**Exports:**
- `data/models/overtake_probability/tcn_overtake_v1.pt`
- `data/models/overtake_probability/tcn_model_config.json`


---

## Step 0: Setup & Imports

In [12]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

OUTPUTS    = repo_root / "notebooks" / "strategy" / "overtake_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "overtake_labeled"
EXPORT_DIR = repo_root / "data" / "models" / "overtake_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [13]:
# ── Sequence config ───────────────────────────────────────────────────────────
SEQ_LEN      = 8          # laps of history per pair
SEQ_FEATURES = ["gap_ahead_s", "pace_delta_s", "drs_window",
                 "tyre_life_diff", "speed_trap_delta"]
N_FEATURES   = len(SEQ_FEATURES)
TARGET       = "overtake"

print(f"Repo root  : {repo_root}")
print(f"Device     : {DEVICE}")
print(f"Seq len    : {SEQ_LEN} laps")
print(f"Features   : {SEQ_FEATURES}")

Repo root  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Device     : cuda
Seq len    : 8 laps
Features   : ['gap_ahead_s', 'pace_delta_s', 'drs_window', 'tyre_life_diff', 'speed_trap_delta']


---

## Step 1 — Load dataset & build temporal sequences

The parquet has one row per (race, lap, pair). We reconstruct the temporal dimension
by grouping each pair `(Year, GP_Name, driver_x, driver_y)`, sorting by `LapNumber`,
and applying a sliding window of length `SEQ_LEN=8`.

Pairs with fewer than 8 laps available are **zero-padded on the left** — the model
sees zeros as "no history yet", which is semantically correct.
The label comes from the **last lap** of each window.


In [ ]:
# ── Step 1 — Load parquet + build sequences ───────────────────────────────────
df = pd.read_parquet(PROCESSED / "overtake_pairs_2023_2025.parquet")

PAIR_KEYS = ["Year", "GP_Name", "driver_x", "driver_y"]
df = df.sort_values(PAIR_KEYS + ["LapNumber"]).reset_index(drop=True)

def build_sequences(group):
    vals = group[SEQ_FEATURES].values.astype(np.float32)
    labs = group[TARGET].values.astype(np.float32)
    X, y = [], []
    for i in range(len(vals)):
        window = vals[max(0, i - SEQ_LEN + 1) : i + 1]
        if len(window) < SEQ_LEN:
            pad    = np.zeros((SEQ_LEN - len(window), N_FEATURES), dtype=np.float32)
            window = np.concatenate([pad, window], axis=0)
        X.append(window)
        y.append(labs[i])
    return np.stack(X), np.array(y, dtype=np.float32)

X_parts, y_parts, year_parts = [], [], []
for keys, grp in df.groupby(PAIR_KEYS):
    X_g, y_g = build_sequences(grp)
    X_parts.append(X_g)
    y_parts.append(y_g)
    year_parts.extend([keys[0]] * len(y_g))

X_all    = np.concatenate(X_parts)
y_all    = np.concatenate(y_parts)
year_all = np.array(year_parts)

print(f"X_all : {X_all.shape}  (samples, seq_len, n_features)")
print(f"y_all : {y_all.shape}  pos={y_all.mean():.3%}")



In [ ]:
# ── Temporal split ────────────────────────────────────────────────────────────
X_train, y_train = X_all[year_all <= 2024], y_all[year_all <= 2024]
X_val,   y_val   = X_all[year_all == 2024], y_all[year_all == 2024]
X_test,  y_test  = X_all[year_all == 2025], y_all[year_all == 2025]

pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

print(f"\nTrain : {X_train.shape}  pos={y_train.mean():.3%}")
print(f"Val   : {X_val.shape}    pos={y_val.mean():.3%}")
print(f"Test  : {X_test.shape}   pos={y_test.mean():.3%}")
print(f"pos_weight: {pos_weight:.2f}")



Train : (18277, 8, 5)  pos=8.918%
Val   : (9047, 8, 5)    pos=8.644%
Test  : (10217, 8, 5)   pos=7.576%
pos_weight: 10.21


---

## Step 2 — Dataset & DataLoader

Standard PyTorch pipeline. The TCN expects input shape `(batch, n_features, seq_len)`
— channels-first — so we transpose the sequence dimension in `__getitem__`.
`pos_weight` passed to `BCEWithLogitsLoss` handles class imbalance.


In [ ]:
# ── Step 2 — Dataset & DataLoader ─────────────────────────────────────────────
class OvertakeSeqDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, seq_len, n_features) → transpose to (N, n_features, seq_len) for Conv1d
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 512

train_ds = OvertakeSeqDataset(X_train, y_train)
val_ds   = OvertakeSeqDataset(X_val,   y_val)
test_ds  = OvertakeSeqDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"Tensor shape  : {next(iter(train_loader))[0].shape}  (batch, n_features, seq_len)")


Train batches : 36
Val   batches : 18
Test  batches : 20
Tensor shape  : torch.Size([512, 5, 8])  (batch, n_features, seq_len)
